In [1]:
import requests
import pandas as pd
import numpy as np
import time

In [2]:
HEADERS = {
    "User-Agent": "your_name your_email@example.com"
}

In [3]:
def get_companyfacts(cik):
    cik = str(cik).zfill(10)
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"

    r = requests.get(url, headers=HEADERS)
    r.raise_for_status()

    return r.json()

In [4]:
CCL= get_companyfacts(815097)

In [5]:
def flatten_companyfacts(companyfacts, symbol=None, taxonomy="us-gaap"):
    rows = []

    cik = companyfacts.get("cik")
    entity_name = companyfacts.get("entityName")

    facts = companyfacts.get("facts", {}).get(taxonomy, {})

    for tag, tag_data in facts.items():
        label = tag_data.get("label")
        description = tag_data.get("description")
        units = tag_data.get("units", {})

        for unit, observations in units.items():
            for obs in observations:
                row = {
                    "symbol": symbol,
                    "cik": cik,
                    "entity_name": entity_name,
                    "tag": tag,
                    "label": label,
                    "description": description,
                    "unit": unit,
                    **obs
                }
                rows.append(row)

    df = pd.DataFrame(rows)

    if df.empty:
        return df

    # Dates
    for col in ["start", "end", "filed"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    # Keep 10-K and 10-Q only
    df = df[df["form"].isin(["10-K", "10-Q"])].copy()

    # Useful period length
    df["period_days"] = (df["end"] - df["start"]).dt.days

    # Clean numeric value
    df["val"] = pd.to_numeric(df["val"], errors="coerce")

    return df

In [6]:
CCL_flattened= flatten_companyfacts(CCL)

In [7]:
CCL_flattened.describe

<bound method NDFrame.describe of       symbol     cik           entity_name  \
0       None  815097  Carnival Corporation   
1       None  815097  Carnival Corporation   
2       None  815097  Carnival Corporation   
3       None  815097  Carnival Corporation   
4       None  815097  Carnival Corporation   
...      ...     ...                   ...   
17785   None  815097  Carnival Corporation   
17786   None  815097  Carnival Corporation   
17787   None  815097  Carnival Corporation   
17788   None  815097  Carnival Corporation   
17789   None  815097  Carnival Corporation   

                                                     tag  \
0                                 AccountsPayableCurrent   
1                                 AccountsPayableCurrent   
2                                 AccountsPayableCurrent   
3                                 AccountsPayableCurrent   
4                                 AccountsPayableCurrent   
...                                                  

In [8]:
def clean_facts(df):
    df = df.copy()

    # Only USD values
    df = df[df["unit"] == "USD"]

    # Only 10-K and 10-Q
    df = df[df["form"].isin(["10-K", "10-Q"])]

    # Drop duplicates (keep latest filing)
    df = (
        df.sort_values(["tag", "start", "end", "filed"])
          .drop_duplicates(subset=["tag", "start", "end"], keep="last")
    )

    return df

In [9]:
CCL_clean_1= clean_facts(CCL_flattened).head(2)

In [10]:
def filter_ttm_window(df, years_back=2, buffer_days=360):
    df = df.copy()

    df["end"] = pd.to_datetime(df["end"], errors="coerce")

    cutoff_date = (
        pd.Timestamp.today().normalize()
        - pd.DateOffset(years=years_back)
        - pd.Timedelta(days=buffer_days)
    )

    df = df[df["end"].notna()].copy()
    df = df[df["end"] >= cutoff_date].copy()

    return df

In [16]:
CCL_clean_1 = clean_facts(CCL_flattened)

CCL_clean_2 = filter_ttm_window(CCL_clean_1)

CCL_clean_2.head()

,symbol,cik,entity_name,tag,label,description,unit,end,val,accn,fy,fp,form,filed,frame,start,period_days
116,None,815097,Carnival Corporation,AccountsPayableCurrent,"Accounts Payable, Current",Carrying value as of the balance sheet date of...,USD,2023-05-31,1.042000e+09,0000815097-23-000051,2023.0,Q2,10-Q,2023-06-28,CY2023Q2I,NaT,NaN
117,None,815097,Carnival Corporation,AccountsPayableCurrent,"Accounts Payable, Current",Carrying value as of the balance sheet date of...,USD,2023-08-31,1.103000e+09,0000815097-23-000066,2023.0,Q3,10-Q,2023-09-29,CY2023Q3I,NaT,NaN
122,None,815097,Carnival Corporation,AccountsPayableCurrent,"Accounts Payable, Current",Carrying value as of the balance sheet date of...,USD,2023-11-30,1.168000e+09,0000815097-25-000007,2024.0,FY,10-K,2025-01-27,CY2023Q4I,NaT,NaN
123,None,815097,Carnival Corporation,AccountsPayableCurrent,"Accounts Payable, Current",Carrying value as of the balance sheet date of...,USD,2024-02-29,1.103000e+09,0000815097-24-000035,2024.0,Q1,10-Q,2024-03-27,CY2024Q1I,NaT,NaN
124,None,815097,Carnival Corporation,AccountsPayableCurrent,"Accounts Payable, Current",Carrying value as of the balance sheet date of...,USD,2024-05-31,1.063000e+09,0000815097-24-000057,2024.0,Q2,10-Q,2024-06-27,CY2024Q2I,NaT,NaN


In [17]:
CCL_clean_1 = clean_facts(CCL_flattened)
CCL_clean_2 = filter_ttm_window(CCL_clean_1)

print(CCL_clean_1.shape)
print(CCL_clean_2.shape)

(8310, 17)
(1475, 17)


,symbol,cik,entity_name,tag,label,description,unit,end,val,accn,fy,fp,form,filed,frame,start,period_days
116,None,815097,Carnival Corporation,AccountsPayableCurrent,"Accounts Payable, Current",Carrying value as of the balance sheet date of...,USD,2023-05-31,1.042000e+09,0000815097-23-000051,2023.0,Q2,10-Q,2023-06-28,CY2023Q2I,NaT,NaN
117,None,815097,Carnival Corporation,AccountsPayableCurrent,"Accounts Payable, Current",Carrying value as of the balance sheet date of...,USD,2023-08-31,1.103000e+09,0000815097-23-000066,2023.0,Q3,10-Q,2023-09-29,CY2023Q3I,NaT,NaN
122,None,815097,Carnival Corporation,AccountsPayableCurrent,"Accounts Payable, Current",Carrying value as of the balance sheet date of...,USD,2023-11-30,1.168000e+09,0000815097-25-000007,2024.0,FY,10-K,2025-01-27,CY2023Q4I,NaT,NaN
123,None,815097,Carnival Corporation,AccountsPayableCurrent,"Accounts Payable, Current",Carrying value as of the balance sheet date of...,USD,2024-02-29,1.103000e+09,0000815097-24-000035,2024.0,Q1,10-Q,2024-03-27,CY2024Q1I,NaT,NaN
124,None,815097,Carnival Corporation,AccountsPayableCurrent,"Accounts Payable, Current",Carrying value as of the balance sheet date of...,USD,2024-05-31,1.063000e+09,0000815097-24-000057,2024.0,Q2,10-Q,2024-06-27,CY2024Q2I,NaT,NaN
